In [1]:
# 安装依赖环境。 
# pymilvus用于操作milvus向量数据库；openai python库用于访问openAI API；requests：HTTP请求库；
# tqdm：进度条可视化库；torch：pytorch深度学习框架，常用于本地运行Embedding模型
!pip install "pymilvus[model]==2.5.10" openai==1.82.0 requests==2.32.3 tqdm==4.67.1 torch==2.7.0

In [2]:
import os
api_key = os.getenv('DEEPSEEK_API_KEY')

In [3]:
# 准备原始的文本数据

from glob import glob
text_lines = []

for file_path in glob("../milvus_docs/en/faq/*.md", recursive=True):
    with open(file_path, "r") as file:
        file_text = file.read()
    text_lines += file_text.split("# ")

len(text_lines)

72

In [4]:
# 准备LLM
from openai import OpenAI
deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
)

In [5]:
# 准备Embedding模型
from pymilvus import model as milvus_model

embedding_model = milvus_model.DefaultEmbeddingFunction()

# 使用刚创建的embedding模型，测试一个文本，并打印其纬度和前几个元素
test_embedding = embedding_model.encode_queries(["I am making my first rag by using milvus and deepseek"])[0]
embedding_dim = len(test_embedding)
print(embedding_dim)
print(test_embedding[:10])

/opt/anaconda3/envs/deepseek-p/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


768
[-0.03385494  0.00070491 -0.00488608  0.01017838  0.01069575 -0.02124413
 -0.0758975   0.003298    0.02508328 -0.00250437]


In [6]:
# 创建collection
from pymilvus import MilvusClient
milvus_client = MilvusClient(uri="./milvus_demo.db")
collection_name = "first_rag_collection"

if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim, # 维度数
    metric_type="IP", # 内积距离，IP表示值越大越相似
    consistency_level="Strong", #一致性级别：Strong总是读到最新数据，可能稍慢。
)

# 插入数据
from tqdm import tqdm

data = []

doc_embeddings = embedding_model.encode_documents(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
TOKENIZERS_PARALLELISM=(true | false) 0/72 [00:00<?, ?it/s]
Creating embeddings: 100%|██████████| 72/72 [00:00<00:00, 492642.56it/s]


{'insert_count': 72, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71], 'cost': 0}

In [7]:
# 检索查询数据
question = "How is data stored in milvus?"

search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode_queries([question]), # 将问题question转为查询向量
    limit=3, # 返回最相似的top3结果
    search_params={"metric_type": "IP", "params":{}}, # 内积距离
    output_fields=["text"], # 返回text字段
)

# 查询的搜索结果
import json
retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

[
    [
        " Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###",
        0.6572663187980652
    ],
    [
        "How does Milvus flush data?\n\nMilvus returns success when inserted data are loaded to t

In [11]:
# 使用LLM获取RAG响应

context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

context


" Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###\nHow does Milvus flush data?\n\nMilvus returns success when inserted data are loaded to the message queue. However, the data are not yet flushed to the dis

In [12]:

SYSTEM_PROMPT = """
Human: 你是一个AI助手。你能够从提供的上下文段落片段中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question>标签括起来的问题。 最后追加原始回答的中文翻译，并用<translate>和<translate>标签标注。
<context>
{context}
<context>
<question>
{question}
</question>
<translated>
</translated>
"""

USER_PROMPT

"\n请使用以下用 <context> 标签括起来的信息片段来回答用 <question>标签括起来的问题。 最后追加原始回答的中文翻译，并用<translate>和<translate>标签标注。\n<context>\n Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###\nHow does Milvus flush data?\n\nMilvus retur

In [14]:
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ]
)

print(response.choices[0].message.content)

Milvus stores data in two main categories: inserted data and metadata.

1. **Inserted Data** (including vector data, scalar data, and collection schemas) are stored as incremental logs in persistent storage. Supported backends include:
   - MinIO
   - AWS S3
   - Google Cloud Storage (GCS)
   - Azure Blob Storage
   - Alibaba Cloud OSS
   - Tencent Cloud Object Storage (COS)

2. **Metadata** (generated internally by Milvus) is stored in etcd, with each module maintaining its own metadata.

For flushing behavior: Data is initially loaded into a message queue before being written to persistent storage as incremental logs. The `flush()` operation forces immediate writing of all queued data to disk.

<translate>
Milvus以两种主要类型存储数据：插入数据和元数据。

1. **插入数据**（包括向量数据、标量数据和集合模式）以增量日志形式存储在持久化存储中，支持的后端包括：
   - MinIO
   - AWS S3
   - 谷歌云存储(GCS)
   - Azure Blob存储
   - 阿里云OSS
   - 腾讯云对象存储(COS)

2. **元数据**（由Milvus内部生成）存储在etcd中，每个模块维护自己的元数据。

关于刷新机制：数据首先被加载到消息队列，随后以增量日志形式写入持久化存储。`flush()`操作会强制立即将所有队列数据写入磁